# CVPO Level 2: Detect → Segment → Classify

## INTRO
This notebook demonstrates CVPO's core value proposition: **chaining multiple models
produces results that no single model can achieve alone.**

We'll use YOLOv8 (detection) + SAM2 (segmentation) + CLIP (classification) on a
capybara image. YOLO can find WHERE the animal is but calls it "bear" (wrong species).
CLIP can identify WHAT it is ("capybara") but doesn't know WHERE. Together through
CVPO's pipeline connectors, you get both correct localization AND correct classification.

## EXPECTATIONS
- See YOLO's closed-vocabulary limitation (it only knows 80 COCO categories)
- See CLIP's open-vocabulary strength (it can classify with any label you provide)
- Understand how the connector between detection and classification bridges the gap
- Run the full 3-model pipeline end-to-end

In [ ]:
# CONFIGURATION
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT / "src"))

IMAGE_PATH = ROOT / "libs" / "capybara.jpg"
LABELS = ["capybara", "beaver", "otter", "bear", "dog"]

## WARMUP — First, see what each model does alone

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from cvpo.core.data_types import PipelineContext
from cvpo.core.stage import StageConfig
from cvpo.stages import YOLOv8Stage, SigLIPStage
from cvpo.pipelines import build_level0_pipeline, build_level2_pipeline

image = np.array(Image.open(IMAGE_PATH).convert("RGB"), dtype=np.uint8)
ctx = PipelineContext(run_id="nb-l2", input_source=str(IMAGE_PATH), frontend="notebook")

# Step 1: YOLO detection alone
detector = YOLOv8Stage(StageConfig(name="det", model_name="yolov8", params={"backend": "yolo"}))
det_result = detector.run(image, ctx)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Show YOLO result
axes[0].imshow(image)
axes[0].set_title("YOLO Detection (alone)")
for d in det_result.detections:
    rect = patches.Rectangle((d.box.x1, d.box.y1), d.box.x2 - d.box.x1, d.box.y2 - d.box.y1,
                              linewidth=2, edgecolor="red", facecolor="none")
    axes[0].add_patch(rect)
    axes[0].text(d.box.x1, d.box.y1 - 5, f"{d.label} ({d.confidence:.2f})",
                color="red", fontsize=12, weight="bold")
axes[0].axis("off")

# Show CLIP classification alone
clip_result = build_level0_pipeline(candidate_labels=LABELS, backend="siglip").run(image, ctx)
axes[1].imshow(image)
axes[1].set_title("CLIP Classification (alone)")
axes[1].text(10, 30, f"Label: {clip_result.classes[0].label} ({clip_result.classes[0].confidence:.2%})",
            color="lime", fontsize=14, weight="bold",
            bbox=dict(boxstyle="round", facecolor="black", alpha=0.7))
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"\nYOLO says: '{det_result.detections[0].label}' — WRONG species (COCO has no 'capybara')")
print(f"CLIP says: '{clip_result.classes[0].label}' — CORRECT species but no location info")

## BUILD — Run the full pipeline: YOLO finds it, connector crops it, CLIP names it

In [ ]:
pipeline = build_level2_pipeline(
    candidate_labels=LABELS,
    detection_backend="yolo",
    segmentation_backend="deterministic",
    classification_backend="siglip",
)

result = pipeline.run(image, ctx)

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.imshow(image)
ax.set_title("CVPO Pipeline: Detect → Segment → Classify")

for cls_entry in result.classes:
    label_text = f"{cls_entry.label} ({cls_entry.confidence:.2%})"
    if cls_entry.is_fallback:
        label_text += " [FALLBACK: no detection, classified full image]"

    if cls_entry.detection_index is not None and cls_entry.detection_index < len(det_result.detections):
        d = det_result.detections[cls_entry.detection_index]
        rect = patches.Rectangle((d.box.x1, d.box.y1), d.box.x2 - d.box.x1, d.box.y2 - d.box.y1,
                                  linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(d.box.x1, d.box.y1 - 5, label_text, color="lime", fontsize=12, weight="bold",
                bbox=dict(boxstyle="round", facecolor="black", alpha=0.7))
    else:
        ax.text(10, 30, label_text, color="yellow", fontsize=14, weight="bold",
                bbox=dict(boxstyle="round", facecolor="black", alpha=0.7))

ax.axis("off")
plt.tight_layout()
plt.show()

print(f"\nPipeline result: {[c.label for c in result.classes]}")
print(f"Fallback flags: {[c.is_fallback for c in result.classes]}")

## ANALYSIS + CONFIRMATION

**What just happened:**
1. **YOLOv8** found the animal and drew a bounding box — but called it "bear" (its closest COCO label)
2. **The connector** cropped the detected region from the original image
3. **CLIP** classified the crop as "capybara" using your open-vocabulary labels

**The core finding:** Neither model alone gives you both WHERE and WHAT correctly.
YOLO knows WHERE (correct box, wrong label). CLIP knows WHAT (correct label, no location).
CVPO's connector bridges the gap — correct localization AND correct classification.

**This pattern** — detect with a fast closed-vocabulary model, then classify with a
precise open-vocabulary model — is the standard two-stage approach from R-CNN
(Girshick et al. 2014), now made composable through CVPO's typed pipeline interface.

| Capability | YOLO Alone | CLIP Alone | CVPO Pipeline |
|------------|-----------|------------|---------------|
| Spatial localization | ✓ | ✗ | ✓ |
| Correct species ID | ✗ (only 80 COCO classes) | ✓ (open vocabulary) | ✓ |

**Next:** Level 3 adds tracking — following objects across video frames with persistent IDs.